In [ ]:
import h5py
import hdf5plugin  # <-- This automatically loads the missing decompression plugins!
import matplotlib.pyplot as plt

with h5py.File('data/tworoom.h5', 'r') as f:
    pixel_data = f['pixels']
    action_data = f['action']
    print("Dataset shape:", pixel_data.shape)

    # This step should work smoothly now without the OSError
    first_image = pixel_data[:20]
    first_action = action_data[:20]

for i in range(first_image.shape[0]):
    # make it bigger
    plt.figure(figsize=(64, 64))
    plt.subplot(1, 20, i + 1)
    plt.imshow(first_image[i])
    plt.axis('off')
    plt.title(f'Action: {first_action[i]}')
plt.show()


In [93]:
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
import pandas as pd
import h5py

model_name = "yujiepan/clip-vit-tiny-random-patch14-336"

encoder_processor = AutoProcessor.from_pretrained(model_name)
encoder = AutoModelForZeroShotImageClassification.from_pretrained(model_name)
# Example using a community-ported ViT-Small checkpoint
processor_model_name = "WinKawaks/vit-small-patch16-224"

predictor_processor = AutoImageProcessor.from_pretrained(processor_model_name)
predictor = AutoModelForImageClassification.from_pretrained(processor_model_name)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 2529.23it/s]


In [ ]:
import torch

with h5py.File("data/tworoom.h5", "r") as f:
    # Load the dataset into a pandas DataFrame
    df = pd.DataFrame({
        'pixels': list(f['pixels'][:64]),
        'actions': list(f['action'][:64])
    })

raw_image = df.iloc[1]['pixels']
raw_image.shape

(224, 224, 3)

In [ ]:
from urllib.request import urlopen
from PIL import Image
import timm

img = Image.open(urlopen(
    'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png'
))

model = timm.create_model(
    'vit_tiny_patch16_224.augreg_in21k_ft_in1k',
    pretrained=True,
    num_classes=0,  # remove classifier nn.Linear
)
model = model.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model)
transforms = timm.data.create_transform(**data_config, is_training=False)

output = model(transforms(img).unsqueeze(0))  # output is (batch_size, num_features) shaped tensor

# or equivalently (without needing to set num_classes=0)

output = model.forward_features(transforms(img).unsqueeze(0))
# output is unpooled, a (1, 197, 192) shaped tensor

output = model.forward_head(output, pre_logits=True)
# output is a (1, num_features) shaped tensor


In [142]:
from encoder import Encoder
batch = torch.randn(16, 3, 224, 224)
encoder = Encoder(input_size=192, output_size=192)
encoder(batch).shape

torch.Size([16, 192])